# Practice Lab: Advanced Reasoning (Lesson 5.3)

Module 5 · 8 exercises · ~120-150 min
**Needs:** google-genai

# Lesson 5.5: Advanced Reasoning — CoT, ToT, ReAct, Reflexion

8 exercises: 2E + 3M + 3H
**Needs:** `pip install google-genai`

In [ ]:
!pip install google-genai -q

In [ ]:
from google import genai
from google.genai import types
import json, os, re, time
from collections import Counter

client = genai.Client(
    api_key=os.environ['GOOGLE_API_KEY'])

def ask(prompt, temp=0.3, system=None):
    resp = client.models.generate_content(
        model='gemini-3-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temp,
            system_instruction=system))
    return resp.text.strip()

---
## Exercise 2 (Easy): Self-Consistency
5 CoT chains at temp=0.7, majority vote.


In [ ]:
# YOUR CODE
problem = (
    'A store sells apples at Rs 40 each. '
    'On weekends, buy-2-get-1-free. '
    'Rahul buys 7 apples on Saturday. '
    'How much does he pay?')

cot_prompt = f"""{problem}
Think step by step, then give the final answer
as a number after 'ANSWER: '."""

# TODO: run 5 chains at temp=0.7
# TODO: extract numeric answer from each
# TODO: majority vote


<details><summary>Solution</summary>


In [ ]:
answers = []
for i in range(5):
    result = ask(cot_prompt, temp=0.7)
    match = re.search(
        r'ANSWER:\s*(?:Rs\.?\s*)?(\d+)',
        result)
    ans = int(match.group(1)) if match else None
    answers.append(ans)
    print(f'  Chain {i+1}: {ans}')

vote = Counter(answers).most_common(1)[0]
print(f'\nMajority vote: {vote[0]} '
      f'({vote[1]}/5 chains agree)')
print(f'Expected: 200')


</details>


---
## Exercise 4 (Medium): Structured CoT Pipeline
GST calculator with <thinking> and <answer> tags.


In [ ]:
# YOUR CODE
invoice_items = [
    {'item': 'Laptop', 'price': 45000,
     'gst_rate': 18},
    {'item': 'Mouse', 'price': 500,
     'gst_rate': 18},
    {'item': 'Notebook', 'price': 50,
     'gst_rate': 12},
]

# TODO: build structured CoT prompt
# TODO: parse <answer> tag as JSON


<details><summary>Solution</summary>


In [ ]:
prompt = f"""Calculate GST for this invoice.
Items: {json.dumps(invoice_items)}
Sale type: Intra-state (Maharashtra)

Reason step by step inside <thinking> tags.
Then give the final breakdown as JSON inside
<answer> tags.

<thinking>
[your step-by-step GST calculation]
</thinking>
<answer>
{{"items": [...], "subtotal": ...,
  "total_cgst": ..., "total_sgst": ...,
  "grand_total": ...}}
</answer>"""

result = ask(prompt)

# Parse <answer> tag
answer_match = re.search(
    r'<answer>(.*?)</answer>', result, re.DOTALL)
if answer_match:
    text = answer_match.group(1).strip()
    text = text.strip('`').replace('json\n', '')
    try:
        data = json.loads(text)
        print('Parsed invoice:')
        print(json.dumps(data, indent=2))
    except json.JSONDecodeError as e:
        print(f'JSON parse error: {e}')
        print(text[:200])
else:
    print('No <answer> tag found')
    print(result[:200])


</details>


---
## Exercise 1 (Easy): CoT vs Direct on 10 GST Problems
Compare direct prompting vs chain-of-thought on GST word problems.


In [ ]:
# YOUR CODE
# Exercise 1: CoT vs Direct on 10 GST Problems
gst_problems = [
    {"q": "A laptop costs Rs 50,000. GST is 18%. "
          "It's an intra-state sale. What is the "
          "CGST and SGST amount?",
     "answer": "CGST=4500, SGST=4500"},
    {"q": "Restaurant bill is Rs 2,000 (AC). "
          "GST rate for AC restaurants is 5% "
          "without ITC. Calculate total bill.",
     "answer": "Total=2100"},
    {"q": "A shirt costs Rs 800 (below Rs 1000 "
          "threshold, 5% GST) and shoes cost "
          "Rs 1,500 (above Rs 1000, 12% GST). "
          "Inter-state sale. Calculate IGST for each.",
     "answer": "Shirt IGST=40, Shoes IGST=180"},
    # TODO: add 7 more problems
]

direct_correct, cot_correct = 0, 0
for p in gst_problems:
    direct = ask(f"{p['q']}\nGive ONLY the answer.")
    cot = ask(f"{p['q']}\nThink step by step, "
              f"then give the final answer.")
    # TODO: check if answers match expected
    print(f"  Q: {p['q'][:50]}...")
    print(f"  Direct: {direct[:60]}")
    print(f"  CoT:    {cot[:60]}")
    print()

# TODO: print accuracy comparison


---
## Exercise 3 (Medium): Reasoning Model Comparison
Compare Standard vs Standard+CoT (and a thinking model) on cost, latency, accuracy.


In [ ]:
# YOUR CODE
# Exercise 3: Reasoning Model Comparison
hard_problem = (
    "A train leaves Station A at 60 km/h. "
    "30 minutes later, another train leaves "
    "Station A at 90 km/h on the same track. "
    "How far from Station A will the second "
    "train catch the first?")
expected = "90 km"

approaches = {}

# (a) Standard + CoT
t0 = time.time()
a = ask(f"{hard_problem}\nThink step by step.")
approaches["Standard+CoT"] = {
    "answer": a[-50:],
    "latency": time.time() - t0,
    "tokens": len(a.split()),
}

# (b) Standard without CoT
t0 = time.time()
b = ask(f"{hard_problem}\nAnswer only.")
approaches["Standard"] = {
    "answer": b[-50:],
    "latency": time.time() - t0,
    "tokens": len(b.split()),
}

# TODO: add Gemini 2.5 Flash with thinking
# (requires enabling thinking in generation_config)

for name, r in approaches.items():
    print(f"  {name:20s}: {r['latency']:.1f}s "
          f"| {r['tokens']} words")
    print(f"    Answer: {r['answer'][:60]}")
    print()


---
## Exercise 5 (Medium): Prompt Decomposition Chain
Chain 4 focused prompts for legal document analysis.


In [ ]:
# YOUR CODE
# Exercise 5: Prompt Decomposition Chain
sample_contract = """
RENTAL AGREEMENT dated 15 March 2025.
Between: Mr. Ravi Kumar (Landlord) and
Ms. Priya Sharma (Tenant).
Property: Flat 302, Green Valley Apartments,
Banjara Hills, Hyderabad.
Rent: Rs 25,000/month, due by 5th of each month.
Security deposit: Rs 1,50,000 (6 months).
Lock-in period: 11 months from date of agreement.
Maintenance: Rs 3,000/month paid by Tenant.
Notice period: 2 months written notice.
Late payment penalty: Rs 500/day after due date.
"""

# Step 1: Extract parties
step1 = ask(f"Extract all parties, dates, and "
            f"property details as JSON:\n"
            f"{sample_contract}")
print("Step 1 (Parties):", step1[:100])

# Step 2: Identify clauses
step2 = ask(f"Given these details: {step1}\n"
            f"Identify all key clauses from:\n"
            f"{sample_contract}")
print("Step 2 (Clauses):", step2[:100])

# Step 3: Flag risks
step3 = ask(f"Given these clauses: {step2}\n"
            f"Flag potential risks for the tenant.")
print("Step 3 (Risks):", step3[:100])

# Step 4: Summary
step4 = ask(f"Write a 5-sentence executive summary "
            f"of this rental agreement.\n"
            f"Parties: {step1}\n"
            f"Risks: {step3}")
print("\nFinal Summary:", step4)

# TODO: compare vs single monolithic prompt


---
## Exercise 6 (Hard): CoVe for Indian Law
Chain-of-Verification to catch hallucinated IPC citations.


In [ ]:
# YOUR CODE
# Exercise 6: CoVe for Indian Law
# Step 1: Generate draft with citations
draft = ask(
    "List 5 specific IPC sections related to "
    "fraud, cheating, and forgery. For each: "
    "section number, title, and key provisions.")
print("DRAFT:", draft[:200])

# Step 2: Generate fact-check questions
questions = ask(
    f"Generate 5 fact-checking questions to "
    f"verify these IPC citations:\n{draft}\n"
    f"For each section, ask: does this section "
    f"number match this description?")
print("\nQUESTIONS:", questions[:200])

# Step 3: Answer INDEPENDENTLY (key!)
# Do NOT show the original draft!
verified = ask(
    f"Answer these questions about Indian "
    f"Penal Code sections. Use your knowledge "
    f"only:\n{questions}")
print("\nVERIFIED:", verified[:200])

# Step 4: Revise
final = ask(
    f"Original citations:\n{draft}\n\n"
    f"Verification results:\n{verified}\n\n"
    f"Produce a corrected list. Mark any "
    f"citations that could not be verified.")
print("\nFINAL:", final)

# TODO: count hallucinated vs verified citations


---
## Exercise 7 (Hard): Cost-Optimized Model Router
Classify query complexity and route to the cheapest sufficient tier.


In [ ]:
# YOUR CODE
# Exercise 7: Cost-Optimized Model Router
def classify_complexity(query):
    """Route query to appropriate model tier."""
    prompt = (
        f"Classify this query's complexity.\n"
        f"Query: '{query}'\n"
        f"Reply with ONLY: simple, medium, or hard\n"
        f"simple = factual lookup, translation\n"
        f"medium = multi-step math, comparison\n"
        f"hard = complex reasoning, planning")
    return ask(prompt, temp=0.0).strip().lower()

# Cost per million tokens (approximate)
costs = {
    "simple": 0.02,   # Flash-Lite
    "medium": 0.15,   # Gemini Flash
    "hard": 1.10,     # Gemini Flash + thinking
}

queries = [
    ("What is the capital of India?", "simple"),
    ("Translate 'hello' to Hindi", "simple"),
    ("Calculate 18% GST on Rs 45,000", "medium"),
    ("Compare 3 investment options for "
     "Rs 10 lakh over 5 years", "hard"),
    # TODO: add 11 more queries
]

total_routed, total_expensive = 0, 0
for query, expected in queries:
    tier = classify_complexity(query)
    routed_cost = costs.get(tier, costs["hard"])
    expensive_cost = costs["hard"]
    total_routed += routed_cost
    total_expensive += expensive_cost
    ok = "Y" if tier == expected else "N"
    print(f"  [{ok}] {tier:7s} {query[:40]}...")

savings = (1 - total_routed / total_expensive) * 100
print(f"\nCost savings: {savings:.0f}%")


---
## Exercise 8 (Challenge): Full Reasoning Benchmark
Benchmark all reasoning techniques on a mixed problem set.


In [ ]:
# YOUR CODE
# Exercise 8: Full Reasoning Benchmark
problems = [
    # Easy (5)
    {"q": "What is 15% of 800?", "a": "120",
     "level": "easy"},
    {"q": "7 * 8 + 3 * 4 = ?", "a": "68",
     "level": "easy"},
    # TODO: add 3 more easy
    # Medium (5)
    {"q": "Shirt Rs 800, 25% off, then 10% off "
          "on discounted price. Final price?",
     "a": "540", "level": "medium"},
    # TODO: add 4 more medium
    # Hard (5)
    {"q": "Train A: 60 km/h. Train B leaves 30 min "
          "later at 90 km/h. How far does B catch A?",
     "a": "90", "level": "hard"},
    # TODO: add 4 more hard
    # Expert (5) - add competition-level problems
]

approaches = {
    "Direct": lambda q: ask(f"{q}\nAnswer only."),
    "Zero-CoT": lambda q: ask(
        f"{q}\nThink step by step. ANSWER:"),
    # TODO: add few-shot CoT, self-consistency,
    #       structured CoT
}

results = {}
for name, fn in approaches.items():
    correct, total_time, total_tokens = 0, 0, 0
    for p in problems:
        t0 = time.time()
        result = fn(p["q"])
        total_time += time.time() - t0
        total_tokens += len(result.split())
        if p["a"] in result:
            correct += 1
    results[name] = {
        "accuracy": correct / len(problems) * 100,
        "latency": total_time / len(problems),
        "tokens": total_tokens,
    }

print(f"{'Approach':<16} {'Acc':>6} {'Lat':>6} "
      f"{'Tokens':>7}")
print("-" * 40)
for name, r in results.items():
    print(f"  {name:<14} {r['accuracy']:>5.0f}% "
          f"{r['latency']:>5.1f}s {r['tokens']:>6d}")


# Hint - self-consistency approach:
def self_consistency(q, n=5):
    answers = []
    for _ in range(n):
        r = ask(f"{q}\nThink step by step. ANSWER:",
                temp=0.7)
        nums = re.findall(r'\d+', r.split("ANSWER:")[-1]
                          if "ANSWER:" in r else r)
        if nums:
            answers.append(nums[-1])
    if not answers:
        return ""
    return Counter(answers).most_common(1)[0][0]
